In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

Our model is not predicting user like this movie and user dislike this movie. But more like user like this movie and user hasn't interacted this movie?

In [13]:
df = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp']).drop(columns=['timestamp'])  # Load your dataset here
df.head()

,user_id,item_id,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [14]:
df["user_id"] = df["user_id"]-1
df["item_id"] = df["item_id"]-1

In [15]:
num_users = df['user_id'].nunique()
num_items = df['item_id'].nunique()

In [16]:
positive_ratings = df[df['rating']>= 4].astype(int)
positive_ratings["label"] = 1
positive_ratings = positive_ratings.drop(columns=['rating'])
positive_ratings.head()

,user_id,item_id,label
5,297,473,1
7,252,464,1
11,285,1013,1
12,199,221,1
16,121,386,1


In [17]:
train_pos, test_pos = train_test_split(
    positive_ratings,
    test_size=0.2,
    random_state=42
)

# Retrieve the positive interactions for each user in the training set
user_positive_items = (
    positive_ratings
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

In [18]:
print(train_pos.head())
print(test_pos.head())
print(user_positive_items[10]) # All items user 10 has positive interactions with.

       user_id  item_id  label
21204      421      447      1
63022      888      195      1
7727       327      330      1
51920      599      448      1
15274      129      587      1
       user_id  item_id  label
36427      451      923      1
63667      786      749      1
73888      715      207      1
16240      161      709      1
9793       289      163      1
{7, 8, 523, 14, 526, 21, 27, 543, 548, 46, 50, 55, 579, 69, 78, 82, 85, 602, 96, 99, 106, 110, 124, 134, 651, 658, 662, 172, 689, 691, 184, 698, 190, 193, 706, 195, 712, 713, 202, 717, 207, 722, 212, 728, 730, 732, 735, 736, 739, 740, 229, 228, 743, 744, 745, 748, 749, 236, 751, 240, 238, 257, 267, 276, 285, 290, 300, 311, 316, 317, 331, 349, 355, 356, 371, 392, 401, 422, 424, 426, 427, 428, 432, 433, 434, 507}


In [19]:
import random

num_negatives = 1 # Number of negative samples to generate per positive sample
train_row = []

# We will iterate through each user-item pair in the training set and generate negative samples for each positive sample. 1:1 ratio
for _, row in train_pos.iterrows():
    user_id = row['user_id']
    positive_item_id = row['item_id']

    # positive
    train_row.append([user_id, positive_item_id, 1])

    # item the user has positively interacted with
    user_positive_items_set = user_positive_items[user_id] # Note that user_positive_items is built from the entire positive_ratings, not just the training set. This is important to avoid sampling negative items that is postiviely interated in the test set.

    # Item that are not in the user's positive interactions
    negative_candidates = list(set(range(num_items)) - user_positive_items_set)

    # sample 1 negative item
    negative_item_id = random.choice(negative_candidates)
    train_row.append([user_id, negative_item_id, 0])

train_df = pd.DataFrame(train_row, columns=['user_id', 'item_id', 'label'])

In [20]:
# look at 421 which is the first id in train_pos, it has 1 positive and 1 negative sample. Same for other ids in train_pos. 
train_df#[train_df['user_id']==10]

,user_id,item_id,label
0,421,447,1
1,421,974,0
2,888,195,1
3,888,659,0
4,327,330,1
...,...,...,...
88595,835,416,0
88596,181,68,1
88597,181,277,0
88598,312,429,1


### We we'll only have our model works on train_df and we'll use test_pos as an answer key. We're not plugging test_set into dataloader like before since that was a binary classification evaluation. What we're doing is retrieval evaluation

In [21]:
class MovieLen100k(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        user_id = self.data.iloc[idx]['user_id']
        item_id = self.data.iloc[idx]['item_id']
        label = self.data.iloc[idx]['label']
        return torch.tensor(user_id, dtype=torch.long), torch.tensor(item_id, dtype=torch.long), torch.tensor(label, dtype=torch.float)

In [22]:
train_set = MovieLen100k(train_df)
train_loader = DataLoader(train_set, batch_size=256, shuffle=True)

In [23]:
class UserTower(nn.Module):
    def __init__(self, num_users, embedding_size, output_size=32):
        super(UserTower, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_size)
        self.fc = nn.Sequential(nn.Linear(embedding_size, 64),
                                nn.ReLU(),
                                nn.Linear(64, output_size))
    def forward(self, user_id):
        user_vector = self.user_embedding(user_id)
        return self.fc(user_vector)

class ItemTower(nn.Module):
    def __init__(self, num_items, embedding_size, output_size=32):
        super(ItemTower, self).__init__() 
        self.item_embedding = nn.Embedding(num_items, embedding_size)      
        self.fc = nn.Sequential(nn.Linear(embedding_size, 64),
                                nn.ReLU(),
                                nn.Linear(64, output_size))
    def forward(self, item_id):
        item_vector = self.item_embedding(item_id)
        return self.fc(item_vector)
    

In [24]:
class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_size):
        super(TwoTowerModel, self).__init__()
        self.user_tower = UserTower(num_users, embedding_size)
        self.item_tower = ItemTower(num_items, embedding_size)


    def forward(self, user_id, item_id):
        user_vector = self.user_tower(user_id)
        item_vector = self.item_tower(item_id)
        return (user_vector * item_vector).sum(1)  # Dot product

In [25]:
model = TwoTowerModel(num_users, num_items, embedding_size=20)
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [26]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for user_id, item_id, label in train_loader:

        optimizer.zero_grad()

        logits = model(user_id, item_id)

        loss = criterion(logits, label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Train Loss: {avg_loss:.4f}"
    )

Epoch 1/10, Train Loss: 0.6383
Epoch 2/10, Train Loss: 0.5065
Epoch 3/10, Train Loss: 0.4608
Epoch 4/10, Train Loss: 0.4432
Epoch 5/10, Train Loss: 0.4342
Epoch 6/10, Train Loss: 0.4266
Epoch 7/10, Train Loss: 0.4226
Epoch 8/10, Train Loss: 0.4181
Epoch 9/10, Train Loss: 0.4143
Epoch 10/10, Train Loss: 0.4095


In [27]:


model.eval()

# All movie IDs
all_items = torch.arange(num_items)

# Compute item embeddings ONCE
with torch.no_grad():
    item_vectors = model.item_tower(all_items)

# Store metrics for every user
all_recalls = []
all_precisions = []
all_ndcgs = []

k = 200

for user_id in range(num_users):

    # --------------------------------------------------
    # 1. Get the user's held-out positive items
    # --------------------------------------------------
    relevant_items = test_pos[
        test_pos["user_id"] == user_id
    ]["item_id"].tolist()

    # Skip users with no positive test interactions
    if len(relevant_items) == 0:
        continue

    # --------------------------------------------------
    # 2. Get items the user already saw during training
    # --------------------------------------------------
    train_items = train_pos[
        train_pos["user_id"] == user_id
    ]["item_id"].tolist()

    # --------------------------------------------------
    # 3. Compute this user's embedding
    # --------------------------------------------------
    user_tensor = torch.tensor(
        [user_id],
        dtype=torch.long
    )

    with torch.no_grad():
        user_vector = model.user_tower(user_tensor)

        # Score user against ALL movies
        scores = user_vector @ item_vectors.T

    # --------------------------------------------------
    # 4. Remove movies the user already saw in training
    # --------------------------------------------------
    scores = scores.squeeze(0)

    scores[train_items] = -float("inf")

    # --------------------------------------------------
    # 5. Get Top-K movies
    # --------------------------------------------------
    top_scores, top_items = torch.topk(
        scores,
        k=k
    )

    top_k = top_items.tolist()

    # --------------------------------------------------
    # 6. Recall@K
    # --------------------------------------------------
    hits = len(
        set(top_k) & set(relevant_items)
    )

    recall = hits / len(relevant_items)

    # --------------------------------------------------
    # 7. Precision@K
    # --------------------------------------------------
    precision = hits / k

    # --------------------------------------------------
    # 8. NDCG@K
    # --------------------------------------------------
    dcg = 0.0

    for rank, item in enumerate(top_k):
        if item in relevant_items:
            dcg += 1 / np.log2(rank + 2)

    ideal_hits = min(len(relevant_items), k)

    idcg = sum(
        1 / np.log2(rank + 2)
        for rank in range(ideal_hits)
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0

    # --------------------------------------------------
    # 9. Save metrics
    # --------------------------------------------------
    all_recalls.append(recall)
    all_precisions.append(precision)
    all_ndcgs.append(ndcg)


# --------------------------------------------------
# 10. Average metrics across users
# --------------------------------------------------

mean_recall = np.mean(all_recalls)
mean_precision = np.mean(all_precisions)
mean_ndcg = np.mean(all_ndcgs)

print(f"Users evaluated: {len(all_recalls)}")
print(f"Mean Recall@{k}: {100*mean_recall:.4f}%")
print(f"Mean Precision@{k}: {100*mean_precision:.4f}%")
print(f"Mean NDCG@{k}: {100*mean_ndcg:.4f}%")

Users evaluated: 921
Mean Recall@200: 61.2013%
Mean Precision@200: 3.7139%
Mean NDCG@200: 28.7339%


## Let's have FAISS in

In [28]:
# Plug in learned item vectors from model above into FAISS for fast retrieval
print(item_vectors.shape)  # Should be (num_items, output_size)

torch.Size([1682, 32])


In [29]:
item_vectors_np = item_vectors.cpu().numpy() # Because FAISS requires numpy arrays, we convert the PyTorch tensor to a numpy array. We also move it to CPU if it's on GPU.


In [30]:
import faiss

index = faiss.IndexFlatIP(32) # 32 is the dimension of the item vectors
index.add(item_vectors_np) # Add item vectors to the index

In [31]:
# Generate User 10's embedding
user_id = 10
user_tensor = torch.tensor([user_id], dtype=torch.long)
with torch.no_grad():
    user_vector = model.user_tower(user_tensor)
    user_vector_np = user_vector.cpu().numpy() # convert to numpy array for FAISS

In [32]:
# Ask FAISS for the top 10 items for User 10
k = 10

scores, item_ids = index.search(
    user_vector_np,
    k
)
print(f'top {k} for user {user_id}: {item_ids}') # this will print "already seen items" for user 10, which is expected since we didn't filter out training items in this FAISS retrieval.

top 10 for user 10: [[ 99  49 180 152  55 356  97 134 215 167]]


In [33]:
print("Scores:", scores)


Scores: [[3.4509034 3.1566355 3.0712318 2.94341   2.934397  2.923642  2.8241136
  2.75054   2.6278276 2.5832095]]


## Now we filter the already seen items in the train_pos

In [34]:
user_id = 10

seen_items = set(
    train_pos[
        train_pos["user_id"] == user_id
    ]["item_id"]
)

print(seen_items)

{257, 134, 7, 8, 523, 651, 14, 526, 401, 276, 21, 662, 27, 285, 543, 290, 548, 422, 424, 172, 428, 46, 300, 432, 433, 434, 691, 689, 50, 55, 184, 311, 698, 316, 317, 190, 193, 706, 579, 195, 69, 712, 713, 202, 331, 717, 78, 207, 82, 722, 728, 602, 730, 732, 349, 735, 96, 736, 99, 739, 229, 740, 743, 744, 745, 106, 356, 228, 749, 238, 751, 240, 371, 507}


In [36]:
k = 100
search_k = 200

scores, item_ids = index.search(user_vector_np, search_k)

filtered_item = []
filtered_scores = []

for score, item_id in zip(scores[0], item_ids[0]):
    if item_id not in seen_items:
        filtered_item.append(item_id)
        filtered_scores.append(score)

    if len(filtered_item) == k:
        break

print(f'top {k} for user {user_id}: {filtered_item}') # Only unseen items are returned now, which is what we want for a real recommendation system.
print(f'Scores: {filtered_scores}')

top 100 for user 10: [np.int64(49), np.int64(180), np.int64(152), np.int64(97), np.int64(215), np.int64(167), np.int64(171), np.int64(175), np.int64(126), np.int64(24), np.int64(271), np.int64(287), np.int64(407), np.int64(274), np.int64(63), np.int64(116), np.int64(292), np.int64(192), np.int64(267), np.int64(0), np.int64(194), np.int64(173), np.int64(208), np.int64(482), np.int64(268), np.int64(95), np.int64(186), np.int64(478), np.int64(10), np.int64(81), np.int64(221), np.int64(514), np.int64(68), np.int64(747), np.int64(426), np.int64(128), np.int64(182), np.int64(136), np.int64(143), np.int64(299), np.int64(404), np.int64(181), np.int64(6), np.int64(203), np.int64(131), np.int64(227), np.int64(178), np.int64(275), np.int64(256), np.int64(209), np.int64(142), np.int64(474), np.int64(210), np.int64(120), np.int64(272), np.int64(284), np.int64(473), np.int64(656), np.int64(187), np.int64(189), np.int64(197), np.int64(94), np.int64(418), np.int64(282), np.int64(11), np.int64(513), np

### Note that the FAISS does not change the quality of retrieval but only the retrieval efficiency, which means the metrics will still be the same as the model above
